# Work with Embeddings: Create

This notebook explores possibilities to profit from embeddings of a collection of sources. The collection is the digital collection of sources of the project [The School of Salamanca](https://salamanca.school/), and the notebook uses multilingual embeddings from OpenAI, Cohere and, eventually, Ollama.

## 0. Preliminaries

In [1]:
# Get info about python version
import sys
print(sys.executable)
print(sys.version)
print(sys.version_info)

/home/awagner/vcs/digicademy/svsal-bertopic/.venv/bin/python
3.11.14 (main, Dec 17 2025, 21:07:37) [Clang 21.1.4 ]
sys.version_info(major=3, minor=11, micro=14, releaselevel='final', serial=0)


## 1. Setup

### 1.1 Install libraries

Instead of using the below python/ipython commands, and in order to make the notebook more declarative/reproducible, we try to define the necessary libraries and environment in a `uv` *project*, i.e. in the [pyproject.toml file](./pyproject.toml) that controls how `uv` manages the `.venv` virtual environment.

According to the [uv documentation](https://docs.astral.sh/uv/concepts/projects/layout/#the-project-environment):

> To run a command in the project environment, use `uv run`. Alternatively the project environment can be activated as normal for a virtual environment.
>
> When `uv run` is invoked, it will create the project environment if it does not exist yet or ensure it is up-to-date if it exists. The project environment can also be explicitly created with `uv sync`.
>
> It is *not* recommended to modify the project environment manually, e.g., with `uv pip install`. For project dependencies, use `uv add` to add a package to the environment.

### 1.2 Load libraries

In [2]:
import os
import dotenv
import itertools
from typing import Any, Dict, List
from datetime import datetime, timezone
import time
import logging

from abc import ABC, abstractmethod
import rich.progress
import rich.console
# import rich.live

import polars as pl
import numpy as np
import pickle
import orjson

import asyncio
import nest_asyncio

import tiktoken
import transformers
import hf_xet
import openai
import cohere
from google import genai
from google.genai import types
import logging
import json

## Model/Provider Configuration Section

In [ ]:
# Input files
file_path_in_text = './in-data/corpus_20260111.csv'  # text and metadata

# Output files
file_path_out = "./out-data"
file_path_prefix = datetime.now().strftime("%Y-%m-%d") + "_"
file_path_out_docs_pickle         = file_path_out + "/" + file_path_prefix + "all_docs.pkl"
file_path_out_docs_parquet        = file_path_out + "/" + file_path_prefix + "all_docs.parquet"
file_path_out_docs_csv            = file_path_out + "/" + file_path_prefix + "all_docs.csv"
file_path_out_embeddings_pickle   = file_path_out + "/" + file_path_prefix + "all_embeddings.pkl"
file_path_out_embeddings_parquet  = file_path_out + "/" + file_path_prefix + "all_embeddings.parquet"
file_path_out_vdb_payloads_jsonl  = file_path_out + "/" + file_path_prefix + "all_embeddings.jsonl"
file_path_out_config_json         = file_path_out + "/" + file_path_prefix + "all_processing_metadata.json"
file_path_out_stats_json          = file_path_out + "/" + file_path_prefix + "all_embedding_statistics.json"

# Check if file_path_out exists, create if not
if not os.path.exists(file_path_out):
    os.makedirs(file_path_out)

# General limits
max_documents = -1    # Set to -1 to process all documents
min_tokens = 10       # Minimum number of tokens for a paragraph to be considered

# Provider configurations
providers_config = {
    # OpenAI Configuration
    "openai": {
        "enabled": True,
        "provider": "openai",
        "model": "text-embedding-3-small",
        "api_spec": "openai",
        "dimensions": 1536,
        "context_limit": 8191,
        "encoding": "cl100k_base",
        "embedding_type": "float", # alternative: base64
        "rate_limit_per_minute": 3000,
        "concurrent_requests": 30,
        "base_url": None,  # Use default OpenAI API URL
        # Multi-window rate limits (optional)
        "rate_limits": {
            "minute": 10000, # Usage tier 4: 10k requests per minute
            # "hour": 180000,  # Estimated
        },
        "time_restrictions": None,  # No time restrictions
        "header_mapping": {},  # OpenAI doesn't provide detailed rate limit headers
        "safety_margin": 0.95
    },

    # Custom OpenAI-compatible Configuration (Academic Cloud)
    "chat-ai-e5-mistral": {
        "enabled": True,
        "provider": "academic-cloud",
        "model": "e5-mistral-7b-instruct",
        "api_spec": "openai",
        "base_url": "https://chat-ai.academiccloud.de/v1",
        "encoding": "cl100k_base",
        "context_limit": 8191,
        "embedding_type": "float",  # alternative: base64
        "rate_limit_per_minute": 60,  # Updated from support email
        "concurrent_requests": 10,  # Reduced given strict limits
        # Multi-window rate limits from support email G#2025050910003915 (2025-05-12)
        # 2 /sec, 60 /min, 3000 /h, 75000 /day, 2000000 /month
        "rate_limits": {
            "minute": 60,
            "hour": 3000,
            "day": 75000,
            "month": 2000000
        },
        # Time restrictions: automated requests only between 19:00-07:00 UTC
        # also per e-Mail from support on 2025-08-12
        "time_restrictions": (19, 7),
        # Header mapping for Academic Cloud
        "header_mapping": {
            "minute": "X-RateLimit-Remaining-Minute",
            "hour": "X-RateLimit-Remaining-Hour",
            "day": "X-RateLimit-Remaining-Day",
            "month": "X-RateLimit-Remaining-Month"
        },
        "safety_margin": 0.95
    },

    # Custom OpenAI-compatible Configuration (Academic Cloud)
    "chat-ai-multilingual-e5": {
        "enabled": True,
        "provider": "academic-cloud",
        "model": "multilingual-e5-large-instruct",
        "api_spec": "openai",
        "base_url": "https://chat-ai.academiccloud.de/v1",
        "context_limit": 512,
        "encoding": "cl100k_base",
        "embedding_type": "float",  # alternative: base64
        "rate_limit_per_minute": 60,  # Updated from support email
        "concurrent_requests": 10,  # Reduced given strict limits
        # Multi-window rate limits from support email G#2025050910003915 (2025-05-12)
        # 2 /sec, 60 /min, 3000 /h, 75000 /day, 2000000 /month
        "rate_limits": {
            "minute": 60,
            "hour": 3000,
            "day": 75000,
            "month": 2000000
        },
        # Time restrictions: automated requests only between 19:00-07:00 UTC
        # also per e-Mail from support on 2025-08-12
        "time_restrictions": (19, 7),
        # Header mapping for Academic Cloud
        "header_mapping": {
            "minute": "X-RateLimit-Remaining-Minute",
            "hour": "X-RateLimit-Remaining-Hour",
            "day": "X-RateLimit-Remaining-Day",
            "month": "X-RateLimit-Remaining-Month"
        },
        "safety_margin": 0.95
    },

    # Custom OpenAI-compatible Configuration (Academic Cloud)
    "chat-ai-qwen3": {
        "enabled": True,
        "provider": "academic-cloud",
        "model": "qwen3-embedding-4b",
        "api_spec": "openai",
        "base_url": "https://chat-ai.academiccloud.de/v1",
        "context_limit": 8191,
        "encoding": "cl100k_base",
        "embedding_type": "float",  # alternative: base64
        "rate_limit_per_minute": 60,  # Updated from support email
        "concurrent_requests": 10,  # Reduced given strict limits
        # Multi-window rate limits from support email G#2025050910003915 (2025-05-12)
        # 2 /sec, 60 /min, 3000 /h, 75000 /day, 2000000 /month
        "rate_limits": {
            "minute": 60,
            "hour": 3000,
            "day": 75000,
            "month": 2000000
        },
        # Time restrictions: automated requests only between 19:00-07:00 UTC
        # also per e-Mail from support on 2025-08-12
        "time_restrictions": (19, 7),
        # Header mapping for Academic Cloud
        "header_mapping": {
            "minute": "X-RateLimit-Remaining-Minute",
            "hour": "X-RateLimit-Remaining-Hour",
            "day": "X-RateLimit-Remaining-Day",
            "month": "X-RateLimit-Remaining-Month"
        },
        "safety_margin": 0.95
    },

    # Cohere Configuration 
    "cohere": {
        "enabled": True,
        "provider": "cohere",
        "model": "embed-v4.0",
        "api_spec": "cohere",
        "dimensions": 1536,
        "context_limit": 128000,
        "input_type": "clustering",  # or "search_document"
        "embedding_types": ["int8"],  # float is the other option
        "rate_limit_per_minute": 2000,
        "concurrent_requests": 20,
        "max_texts_per_call": 96,
        # Multi-window rate limits (optional)
        "rate_limits": {
            "minute": 2000,
        },
        "time_restrictions": None,  # No time restrictions
        "header_mapping": {},  # Cohere doesn't provide detailed rate limit headers
        "safety_margin": 0.95
    },

    # Google Gemini Configuration 
    "gemini": {
        "enabled": True,
        "provider": "google",
        "model": "gemini-embedding-001",
        "api_spec": "google",
        "dimensions": 1536,     # Supports 768, 1536, or 3072; defaults to 3072 if not specified
        "context_limit": 2048,  # Token limit for gemini-embedding-001
        "task_type": "SEMANTIC_SIMILARITY",  # Options: SEMANTIC_SIMILARITY, CLASSIFICATION, CLUSTERING, RETRIEVAL_DOCUMENT, RETRIEVAL_QUERY, etc.
        "rate_limit_per_minute": 1500,  # Conservative limit based on Gemini API quotas
        "concurrent_requests": 20,
        # Multi-window rate limits (optional)
        "rate_limits": {
            "minute": 3000, # Free tier allows 100 requests per minute, paid tiers higher
            # free tier allows 1000 requests per day, paid tiers are unlimited
        },
        "time_restrictions": None,  # No time restrictions
        "header_mapping": {},  # Google doesn't provide detailed rate limit headers
        "safety_margin": 0.95
    }
}

# Select which providers to use (can specify multiple)
active_providers = ["openai", "chat-ai-e5-mistral", "chat-ai-multilingual-e5", "chat-ai-qwen3", "cohere", "gemini"]
active_providers = [p for p in active_providers if p in providers_config and providers_config[p]["enabled"]]


#### Provider Configuration Notes:

Each provider configuration now supports the following rate limiting options:

```python
{
    # ... existing config ...
    
    # Multi-window rate limits (optional)
    "rate_limits": {
        "minute": 60,    # requests per minute
        "hour": 3000,    # requests per hour
        "day": 75000,    # requests per day
        "month": 2000000 # requests per month
    },
    
    # Time restrictions (start_hour, end_hour) in UTC, e.g., (19, 7) = 19:00-07:00
    "time_restrictions": (19, 7),  # or None for no restrictions
    
    # Header mapping for synchronization
    "header_mapping": {
        "minute": "X-RateLimit-Remaining-Minute",
        "hour": "X-RateLimit-Remaining-Hour",
        "day": "X-RateLimit-Remaining-Day",
        "month": "X-RateLimit-Remaining-Month"
    },
    
    # Safety margin (use 95% of actual limits)
    "safety_margin": 0.95
}
```

**Backwards Compatibility**: If `rate_limits` is not specified, the system falls back to the simple `rate_limit_per_minute` approach.


## API Keys Setup

In [4]:
# Find the .env file and load it
dotenv.load_dotenv(dotenv.find_dotenv())

# Get API keys with fallbacks
api_keys = {
    "openai": os.getenv("OPENAI_API_KEY", ""),
    "academic-cloud": os.getenv("ACADEMIC_CLOUD_API_KEY") or os.getenv("OPENAI_API_KEY", ""),
    "cohere": os.getenv("COHERE_API_KEY", ""),
    "google": os.getenv("GEMINI_API_KEY", ""),
}

# Check if the required API keys are set
for provider in active_providers:
    if not api_keys[providers_config[provider]["provider"]]:
        raise ValueError(f"API key for {provider} is not set. Please set {provider.upper()}_API_KEY in your .env file.")

## Utility Functions

### Logging Configuration

Configure structured logging for the embedding generation process.

In [5]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler()]
)
# Set httpx to only show warnings and errors (not every successful request)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("openai._base_client").setLevel(logging.WARNING)
logger = logging.getLogger(__name__)

### Statistics Tracking Class

Class to track embedding generation statistics per provider.

In [6]:
class EmbeddingStatistics:
    """Track statistics for embedding generation per provider."""
    
    def __init__(self, provider_name: str):
        self.provider_name = provider_name
        self.total = 0
        self.success = 0
        self.failed = 0
        self.failed_docs = []
        self.rate_limit_hits = 0
        self.start_time = None
        self.end_time = None
    
    def start(self):
        """Mark the start of processing."""
        self.start_time = time.time()
        logger.info(f"Provider {self.provider_name} started processing")
    
    def complete(self):
        """Mark the completion of processing."""
        self.end_time = time.time()
        logger.info(f"Provider {self.provider_name} completed processing")
    
    def record_success(self, doc_id: str):
        """Record a successful embedding."""
        self.total += 1
        self.success += 1
    
    def record_failure(self, doc_id: str, error: str):
        """Record a failed embedding."""
        self.total += 1
        self.failed += 1
        self.failed_docs.append({"doc_id": doc_id, "error": error})
        logger.warning(f"Provider {self.provider_name} - Document {doc_id} failed: {error}")
    
    def record_rate_limit(self):
        """Record a rate limit hit."""
        self.rate_limit_hits += 1
        logger.warning(f"Provider {self.provider_name} - Rate limit hit (total: {self.rate_limit_hits})")
    
    def get_processing_time(self) -> float:
        """Get processing time in seconds."""
        if self.start_time and self.end_time:
            return self.end_time - self.start_time
        return 0.0
    
    def get_success_rate(self) -> float:
        """Get success rate as percentage."""
        if self.total == 0:
            return 0.0
        return (self.success / self.total) * 100.0
    
    def to_dict(self) -> dict:
        """Convert statistics to dictionary."""
        return {
            "provider_name": self.provider_name,
            "total": self.total,
            "success": self.success,
            "failed": self.failed,
            "success_rate": round(self.get_success_rate(), 2),
            "processing_time": round(self.get_processing_time(), 2),
            "rate_limit_hits": self.rate_limit_hits,
            "failed_docs": self.failed_docs
        }
    
    def print_summary(self):
        """Print a summary of the statistics."""
        processing_time = self.get_processing_time()
        success_rate = self.get_success_rate()
        
        print(f"\n{self.provider_name}:")
        print(f"  Total: {self.total}")
        print(f"  Success: {self.success} ({success_rate:.1f}%)")
        print(f"  Failed: {self.failed} ({100-success_rate:.1f}%)")
        print(f"  Time: {processing_time:.1f}s")
        if self.rate_limit_hits > 0:
            print(f"  Rate limit hits: {self.rate_limit_hits}")

### Rate Limiting Implementation

This section implements a sophisticated multi-window rate limiting system that supports:

#### Features:

1. **Multi-Window Rate Limiting**
   - Supports minute, hour, day, and month time windows per provider
   - Uses token bucket algorithm with per-window buckets
   - Blocks requests only when ALL windows can provide tokens (conjunctive enforcement)
   - Thread-safe with asyncio.Lock

2. **Header-Based Synchronization**
   - After each API response, reads provider rate limit headers
   - Updates local token bucket state from headers (headers are ground truth)
   - Handles different header formats per provider (OpenAI, Academic Cloud, Cohere)

3. **429 Handling**
   - Respects `Retry-After` header when receiving 429 responses
   - Includes exponential backoff with header-based override
   - Tracks consecutive 429s for monitoring

4. **Time-of-Day Restrictions** (Academic Cloud specific)
   - Only sends automated requests between 19:00-07:00 UTC
   - Configurable time window per provider
   - Automatic waiting when outside allowed window

5. **Safety Margins**
   - Uses 95% of actual limits by default to prevent edge cases
   - Configurable per provider

#### Classes:

- **TokenBucket**: Implements token bucket algorithm with automatic refilling
- **HeaderBasedRateLimiter**: Manages multiple token buckets and enforces all constraints


In [7]:
class TokenBucket:
    """Token bucket implementation for rate limiting."""
    def __init__(self, capacity: float, refill_per_second: float):
        self.capacity = float(capacity)
        self.refill_per_second = float(refill_per_second)
        self._tokens = float(capacity)
        self._last = time.monotonic()
    
    def _refill(self):
        now = time.monotonic()
        elapsed = now - self._last
        self._last = now
        self._tokens = min(self.capacity, self._tokens + elapsed * self.refill_per_second)
    
    def can_consume(self, amount: float = 1.0) -> bool:
        self._refill()
        return self._tokens >= amount
    
    def consume(self, amount: float = 1.0) -> bool:
        self._refill()
        if self._tokens >= amount:
            self._tokens -= amount
            return True
        return False
    
    def set_tokens(self, value: float):
        """Update tokens from authoritative source (headers)."""
        self._tokens = min(float(value), self.capacity)
        self._last = time.monotonic()
    
    def get_tokens(self) -> float:
        """Get current token count."""
        self._refill()
        return self._tokens

class HeaderBasedRateLimiter:
    """Multi-window rate limiter with header-based synchronization."""
    def __init__(self, windows: Dict[str, tuple],
                 time_restrictions: tuple,
                 safety_margin: float = 0.95):
        """
        Initialize rate limiter with multiple time windows.
        
        Args:
            windows: {name: (limit, window_seconds)}
            time_restrictions: (start_hour, end_hour) UTC, e.g., (19, 7) means 19:00-07:00
            safety_margin: Fraction of limit to use (0.95 = use 95% of actual limit)
        """
        self.buckets = {}
        for name, (limit, window_sec) in windows.items():
            adjusted_limit = limit * safety_margin
            refill_rate = adjusted_limit / window_sec
            self.buckets[name] = TokenBucket(adjusted_limit, refill_rate)
        
        self.time_restrictions = time_restrictions
        self.lock = asyncio.Lock()
        self.last_429_time = 0
        self.consecutive_429s = 0
    
    def _is_allowed_time(self) -> bool:
        """Check if current time is within allowed time window."""
        if not self.time_restrictions:
            return True
        start_hour, end_hour = self.time_restrictions
        current_hour = datetime.now(timezone.utc).hour
        if start_hour > end_hour:  # wraps midnight
            return current_hour >= start_hour or current_hour < end_hour
        else:
            return start_hour <= current_hour < end_hour
    
    async def acquire(self, amount: float = 1.0):
        """Acquire tokens from all buckets (blocks until available)."""
        async with self.lock:
            # Log rate limiter activity (optional, can be disabled)
            logging.debug(f"Rate limiter acquiring: {self.get_status()}")
            # Check time restrictions
            while not self._is_allowed_time():
                wait_minutes = 5
                print(f"Outside allowed time window. Waiting {wait_minutes} minutes...")
                await asyncio.sleep(wait_minutes * 60)
            
            # Wait until all buckets can provide tokens
            while True:
                if all(bucket.can_consume(amount) for bucket in self.buckets.values()):
                    for bucket in self.buckets.values():
                        bucket.consume(amount)
                    return
                
                # Calculate wait time based on most restrictive bucket
                max_wait = 0
                for name, bucket in self.buckets.items():
                    if not bucket.can_consume(amount):
                        shortage = amount - bucket._tokens
                        wait_time = shortage / max(bucket.refill_per_second, 1e-6)
                        max_wait = max(max_wait, wait_time)
                
                # Wait with a small buffer
                await asyncio.sleep(min(max_wait + 0.1, 1.0))
    
    def update_from_headers(self, headers: dict, header_mapping: dict):
        """
        Update buckets from response headers.
        
        Args:
            headers: Response headers from API
            header_mapping: {window_name: header_key}
        """
        if not header_mapping:
            return
            
        for window_name, header_key in header_mapping.items():
            if window_name in self.buckets and header_key in headers:
                try:
                    remaining = float(headers[header_key])
                    self.buckets[window_name].set_tokens(remaining)
                except (ValueError, TypeError):
                    pass
    
    async def handle_429(self, headers: dict):
        """Handle 429 by respecting Retry-After header."""
        self.consecutive_429s += 1
        self.last_429_time = time.time()
        
        retry_after = headers.get('Retry-After', headers.get('RateLimit-Reset'))
        if retry_after:
            try:
                wait_seconds = float(retry_after)
                print(f"Rate limited (429). Waiting {wait_seconds} seconds...")
                await asyncio.sleep(wait_seconds)
                return
            except (ValueError, TypeError):
                pass
        
        # Exponential backoff if no header or invalid
        backoff = min(60, 2 ** self.consecutive_429s)
        print(f"Rate limited (429). Backing off for {backoff} seconds...")
        await asyncio.sleep(backoff)
    
    def reset_429_counter(self):
        """Reset 429 counter after successful requests."""
        self.consecutive_429s = 0
    
    def get_status(self) -> dict:
        """Get current status of all buckets for monitoring."""
        return {name: bucket.get_tokens() for name, bucket in self.buckets.items()}

# Batched (a function from Python's own itertools cookbook) breaks up a sequence into chunks
def batched(iterable, n):
    """Batch data into tuples of length n. The last batch may be shorter."""
    # batched('ABCDEFG', 3) --> ABC DEF G
    if n < 1:
        raise ValueError('n must be at least one')
    it = iter(iterable)
    while (batch := tuple(itertools.islice(it, n))):
        yield batch

# Tracked semaphore class to limit concurrent requests
class TrackedSemaphore:
    def __init__(self, limit: int):
        self._semaphore = asyncio.Semaphore(limit)
        self._current_tasks = 0
        self._limit = limit

    @property
    def current_tasks(self):
        return self._current_tasks

    @property
    def limit(self):
        return self._limit

    # Context Manager Support
    async def __aenter__(self):
        await self.acquire()
        return self

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        self.release()

    # Direct acquire/release support
    async def acquire(self):
        await self._semaphore.acquire()
        self._current_tasks += 1

    def release(self):
        self._current_tasks -= 1
        if self._current_tasks < 0:
            raise ValueError("Semaphore counter went negative. This should not happen.")
        self._semaphore.release()

### Saving functions

In [8]:
def save_dict_pickle(e: dict, path: str) -> None:
    """Save a dictionary to a pickle file."""
    print("Saving dictionary pickle to disk ...")
    with open(path, "wb") as fOut:
        pickle.dump(e, fOut)
    print(f"Saved dictionary with {len(e)} providers to {path}\n")

def save_dict_parquet(e: dict, path: str) -> None:
    """Save a dictionary to a parquet file."""
    print("Saving dictionary parquet to disk ...")
    with open(path, "wb") as fOut:
        pl.DataFrame(e).write_parquet(path)
    print(f"Saved dictionary with {len(e)} providers to {path}\n")

def save_df_pickle(docs: pl.DataFrame, path: str) -> None:
    """Save a dataframe to a pickle file."""
    print("Saving dataframe pickle to disk ...")
    with open(path, "wb") as fOut:
        pickle.dump(docs, fOut)
    print(f"Saved {len(docs)} dataframe entries to {path}\n")

def save_df_parquet(docs: pl.DataFrame, path: str) -> None:
    """Save a dataframe to a parquet file."""
    print("Saving dataframe parquet to disk ...")
    with open(path, "wb") as fOut:
        pl.DataFrame(docs).write_parquet(path)
    print(f"Saved {len(docs)} dataframe entries to {path}\n")

def save_metadata_json(metadata: dict, path: str) -> None:
    """Save processing metadata to a JSON file."""
    print("Saving processing metadata to disk ...")
    with open(path, 'wb') as fOut:
        fOut.write(
            orjson.dumps(
                metadata,
                option=orjson.OPT_INDENT_2 | orjson.OPT_APPEND_NEWLINE
            )
        )
    print(f"Saved processing metadata to {path}\n")

## Provider Class Implementation

In [9]:
class EmbeddingProvider(ABC):
    """Abstract base class for embedding providers."""

    config: Dict[str, Any]
    api_key: str
    delay: float
    rate_limiter: 'HeaderBasedRateLimiter' = None
    header_mapping: dict = {}

    @abstractmethod
    async def initialize(self) -> None:
        """Initialize the provider client."""
        pass

    @abstractmethod
    async def create_embedding(self, text: str, semaphore: TrackedSemaphore) -> List[float]:
        """Create an embedding for the given text."""
        pass

    @abstractmethod
    def get_identifier(self) -> str:
        """Return a unique identifier for this provider and model."""
        pass
    
    @abstractmethod
    def get_tokenizer(self, text: str) -> List[int]:
        """Tokenize the input text."""
        pass
    
    @abstractmethod
    def chunk_text(self, text: str) -> List[str]:
        """Chunk text if it exceeds the context window."""
        pass

class OpenAIProvider(EmbeddingProvider):
    """OpenAI embedding provider implementation."""

    def __init__(self, config: Dict[str, Any], api_key: str, shared_rate_limiter=None):
        self.config = config
        self.api_key = api_key
        self.client = None
        self.delay = config["concurrent_requests"] * 60.0 / config["rate_limit_per_minute"]
        self.rate_limiter = shared_rate_limiter
        self.header_mapping = config.get("header_mapping", {})

    async def initialize(self) -> None:
        self.client = openai.AsyncOpenAI(
            api_key=self.api_key,
            base_url=self.config["base_url"]
        )

    async def create_embedding(self, text: str, semaphore: TrackedSemaphore) -> List[float]:
        if not self.client:
            await self.initialize()

        # Acquire rate limiter token first (before semaphore)
        if self.rate_limiter:
            await self.rate_limiter.acquire(1.0)

        async with semaphore:
            # Adjust delay based on the number of parallel requests underway
            current_parallel_requests = semaphore.current_tasks
            adjusted_delay = self.delay * max(1, current_parallel_requests)
            await asyncio.sleep(adjusted_delay)

            # Prepare text chunks - this provider can handle multiple texts in one call
            input_chunks = self.chunk_text(text)

            # Prepare parameters for API requests
            params = {
                "input": input_chunks,
                "model": self.config["model"],
                "encoding_format": self.config["embedding_type"]
            }
            if "dimensions" in self.config:
                params["dimensions"] = self.config["dimensions"]

            # Do the API requests
            retry_count = 3
            for attempt in range(retry_count):
                try:
                    response = await self.client.embeddings.create(**params)
                    
                    # Update rate limiter from response headers if available
                    if self.rate_limiter and hasattr(response, 'headers'):
                        self.rate_limiter.update_from_headers(
                            dict(response.headers) if response.headers else {},
                            self.header_mapping
                        )
                        self.rate_limiter.reset_429_counter()
                    
                    embeddings = [response.data[i].embedding for i in range(len(response.data))]

                    # If multiple chunks, average the embeddings
                    if len(embeddings) > 1:
                        print(f"Warning: Received {len(embeddings)} embeddings from OpenAI. Averaging them.")
                        # Calculate weighted average by text length
                        weights = [len(chunk) for chunk in input_chunks]
                        avg_embedding = np.average(embeddings, axis=0, weights=weights)
                        # Normalize
                        avg_embedding = avg_embedding / np.linalg.norm(avg_embedding)
                        return avg_embedding.tolist()
                    else:
                        return embeddings[0]

                except Exception as e:
                    # Check if it's a 429 error
                    if hasattr(e, 'status_code') and e.status_code == 429:
                        headers = dict(e.response.headers) if hasattr(e, 'response') and hasattr(e.response, 'headers') else {}
                        if self.rate_limiter:
                            await self.rate_limiter.handle_429(headers)
                        # Don't count this as a retry attempt
                        continue
                    
                    if attempt < retry_count - 1:
                        await asyncio.sleep(2 ** attempt)  # Exponential backoff
                    else:
                        raise

    def get_identifier(self) -> str:
        return f"{self.config['provider']}_{self.config['model']}"

    def get_tokenizer(self, text: str) -> List[int]:
        # This depends on the model in use, examples are tiktoken or HuggingFace's AutoTokenizer
        if self.config["model"] == "e5-mistral-7b-instruct":
            tokenizer = transformers.AutoTokenizer.from_pretrained('intfloat/e5-mistral-7b-instruct')
            tokens = tokenizer(text, return_tensors="pt", truncation=True, max_length=self.config["context_limit"])
            return tokens["input_ids"].squeeze().tolist()
        elif self.config["model"] == "multilingual-e5-large-instruct":
            tokenizer = transformers.AutoTokenizer.from_pretrained('intfloat/multilingual-e5-large-instruct')
            tokens = tokenizer(text, return_tensors="pt", truncation=True, max_length=self.config["context_limit"])
            return tokens["input_ids"].squeeze().tolist()
        elif self.config["model"] == "qwen3-embedding-4b":
            tokenizer = transformers.AutoTokenizer.from_pretrained('Qwen/Qwen3-Embedding-4B')
            tokens = tokenizer(text, return_tensors="pt", truncation=True, max_length=self.config["context_limit"])
            return tokens["input_ids"].squeeze().tolist()
        else:
            tokenizer = tiktoken.get_encoding(self.config["encoding"])
            return tokenizer.encode(text)

    def chunk_text(self, text: str) -> List[str]:
        # This depends on the model in use, examples are tiktoken or HuggingFace's AutoTokenizer
        tokenizer = None
        tokens = []
        is_hf_tokenizer = False
        
        if self.config["model"] == "e5-mistral-7b-instruct":
            tokenizer = transformers.AutoTokenizer.from_pretrained('intfloat/e5-mistral-7b-instruct')
            # Don't truncate during tokenization - we want to see the full token count
            _tokens = tokenizer(text, return_tensors="pt", truncation=False, add_special_tokens=False)
            tokens = _tokens["input_ids"].squeeze().tolist()
            is_hf_tokenizer = True
        elif self.config["model"] == "multilingual-e5-large-instruct":
            tokenizer = transformers.AutoTokenizer.from_pretrained('intfloat/multilingual-e5-large-instruct')
            # Don't truncate during tokenization - we want to see the full token count
            _tokens = tokenizer(text, return_tensors="pt", truncation=False, add_special_tokens=False)
            tokens = _tokens["input_ids"].squeeze().tolist()
            is_hf_tokenizer = True
        elif self.config["model"] == "qwen3-embedding-4b":
            tokenizer = transformers.AutoTokenizer.from_pretrained('Qwen/Qwen3-Embedding-4B')
            # Don't truncate during tokenization - we want to see the full token count
            _tokens = tokenizer(text, return_tensors="pt", truncation=False, add_special_tokens=False)
            tokens = _tokens["input_ids"].squeeze().tolist()
            is_hf_tokenizer = True
        else:
            tokenizer = tiktoken.encoding_for_model(self.config["model"])
            tokens = tokenizer.encode(text)

        # Now chunk the tokens based on context limit
        # For HuggingFace tokenizers, reserve space for special tokens
        if is_hf_tokenizer:
            # Reserve space for special tokens (e.g., [CLS], [SEP])
            num_special_tokens = tokenizer.num_special_tokens_to_add(pair=False)
            effective_limit = self.config["context_limit"] - num_special_tokens
        else:
            effective_limit = self.config["context_limit"]
        
        chunks = []
        for chunk in batched(tokens, effective_limit):
            if is_hf_tokenizer:
                # For HF tokenizers, decode without special tokens and let the API add them
                chunks.append(tokenizer.decode(chunk, skip_special_tokens=True))
            else:
                chunks.append(tokenizer.decode(chunk))

        return chunks

class CohereProvider(EmbeddingProvider):
    """Cohere embedding provider implementation."""

    def __init__(self, config: Dict[str, Any], api_key: str, shared_rate_limiter=None):
        self.config = config
        self.api_key = api_key
        self.client = None
        self.delay = config["concurrent_requests"] * 60.0 / config["rate_limit_per_minute"]
        self.rate_limiter = shared_rate_limiter
        self.header_mapping = config.get("header_mapping", {})

    async def initialize(self) -> None:
        # self.client = cohere.AsyncClient(api_key=self.api_key)
        self.client = cohere.ClientV2(api_key=self.api_key)

    async def create_embedding(self, text: str, semaphore: TrackedSemaphore) -> List[float]:
        if not self.client:
            await self.initialize()

        # Acquire rate limiter token first (before semaphore)
        if self.rate_limiter:
            await self.rate_limiter.acquire(1.0)

        async with semaphore:
            # Adjust delay based on the number of parallel requests underway
            current_parallel_requests = semaphore.current_tasks
            adjusted_delay = self.delay * max(1, current_parallel_requests)
            await asyncio.sleep(adjusted_delay)

            # Prepare text chunks - this provider can handle multiple texts in one call
            input_chunks = self.chunk_text(text)

            # Prepare parameters for API requests
            params = {
                "texts": input_chunks,
                "model": self.config["model"],
                "input_type": self.config["input_type"],
                "embedding_types": self.config["embedding_types"]
            }
            if "dimensions" in self.config:
                params["output_dimension"] = self.config["dimensions"]

            # Do the API requests
            retry_count = 3
            for attempt in range(retry_count):
                try:
                    response = self.client.embed(**params)
                    
                    # Update rate limiter from response headers if available
                    if self.rate_limiter and hasattr(response, 'headers'):
                        self.rate_limiter.update_from_headers(
                            dict(response.headers) if response.headers else {},
                            self.header_mapping
                        )
                        self.rate_limiter.reset_429_counter()
                    
                    # The embedding might process multiple numerical data types, but for now we only care about the first one
                    extract_embedding_type = self.config["embedding_types"][0]
                    embeddings = getattr(response.embeddings, extract_embedding_type)

                    # If the response is empty, raise an error
                    if not embeddings:
                        raise ValueError("Received empty embeddings from Cohere.")
                    # If the response is not a list, raise an error
                    if not isinstance(embeddings, list):
                        print(f"Warning: Received non-list embeddings from Cohere. Type: {type(embeddings)}")
                        print(f"Received: {embeddings}")
                        raise ValueError("Received non-list embeddings from Cohere.")
                    # If the response is not a list of lists, raise an error
                    if not all(isinstance(embedding, list) for embedding in embeddings):
                        raise ValueError("Received non-list of lists embeddings from Cohere.")
                    # If the response is not a list of lists of embedding_type numbers, raise an error
                    if extract_embedding_type == "float":
                        if not all(isinstance(embedding, float) for embedding in itertools.chain.from_iterable(embeddings)):
                            raise ValueError("Received non-list of lists of float embeddings from Cohere.")
                    elif extract_embedding_type == "int8":
                        if not all(isinstance(embedding, int) for embedding in itertools.chain.from_iterable(embeddings)):
                            raise ValueError("Received non-list of lists of int8 embeddings from Cohere.")
                    # If the response is not a list of lists of the correct length, print a warning
                    if not all(len(embedding) == self.config["dimensions"] for embedding in embeddings):
                        print(f"Warning: Received embeddings of incorrect length from Cohere. Expected {self.config['dimensions']}, got {len(embeddings[0])}.")
                    
                    # If multiple chunks, average the embeddings
                    if len(embeddings) > 1:
                        # Calculate weighted average by text length
                        weights = [len(chunk) for chunk in input_chunks]
                        avg_embedding = np.average(a=embeddings, axis=0, weights=weights)
                        # Normalize
                        avg_embedding = avg_embedding / np.linalg.norm(avg_embedding)
                        return avg_embedding.tolist()
                    else:
                        return embeddings[0]

                except Exception as e:
                    # Check if it's a 429 error
                    if hasattr(e, 'status_code') and e.status_code == 429:
                        headers = dict(e.response.headers) if hasattr(e, 'response') and hasattr(e.response, 'headers') else {}
                        if self.rate_limiter:
                            await self.rate_limiter.handle_429(headers)
                        # Don't count this as a retry attempt
                        continue
                    
                    if attempt < retry_count - 1:
                        await asyncio.sleep(2 ** attempt)  # Exponential backoff
                    else:
                        raise

    def get_identifier(self) -> str:
        return f"{self.config['provider']}_{self.config['model']}_{self.config['input_type']}"

    def get_tokenizer(self, text: str) -> List[int]:
        return self.client.tokenize(text=text, model=self.config["model"])

    def chunk_text(self, text: str) -> List[str]:
        # Split text if it exceeds max length
        # Cohere can handle long texts, but has a limit on tokens per call
        tokens = self.client.tokenize(text=text, model=self.config["model"]).tokens
        if len(tokens) <= self.config["context_limit"]:
            return [text]

        # Simplified chunking by character count (Cohere handles larger contexts)
        # We use a rough approximation: avg 4 chars per token
        avg_chars_per_token = 4
        chars_per_chunk = (self.config["context_limit"] * avg_chars_per_token)

        # Split into chunks
        chunks = []
        for i in range(0, len(text), chars_per_chunk):
            chunks.append(text[i:i + chars_per_chunk])

        return chunks

class GoogleProvider(EmbeddingProvider):
    """Google Gemini embedding provider implementation."""
    
    def __init__(self, config: Dict[str, Any], api_key: str, shared_rate_limiter=None):
        self.config = config
        self.api_key = api_key
        self.client = None
        self.delay = config["concurrent_requests"] * 60.0 / config["rate_limit_per_minute"]
        self.rate_limiter = shared_rate_limiter
        self.header_mapping = config.get("header_mapping", {})

    async def initialize(self) -> None:
        """Initialize the Google Gemini client."""
        self.client = genai.Client(api_key=self.api_key)

    async def create_embedding(self, text: str, semaphore: TrackedSemaphore) -> List[float]:
        """Create embedding using Google Gemini API."""
        if not self.client:
            await self.initialize()

        # Acquire rate limiter token first (before semaphore)
        if self.rate_limiter:
            await self.rate_limiter.acquire(1.0)

        async with semaphore:
            # Adjust delay based on the number of parallel requests underway
            current_parallel_requests = semaphore.current_tasks
            adjusted_delay = self.delay * max(1, current_parallel_requests)
            await asyncio.sleep(adjusted_delay)

            # Prepare text chunks
            input_chunks = self.chunk_text(text)
            
            config_params = {}
            if "dimensions" in self.config:
                config_params["output_dimensionality"] = self.config["dimensions"]
            
            # Set task type if provided
            if "task_type" in self.config:
                config_params["task_type"] = self.config["task_type"]
            
            embed_config = types.EmbedContentConfig(**config_params) if config_params else None

            # Do the API requests
            retry_count = 3
            for attempt in range(retry_count):
                try:
                    # Google API - pass chunks as list of strings
                    response = self.client.models.embed_content(
                        model=self.config["model"],
                        contents=input_chunks,
                        config=embed_config
                    )

                    # Update rate limiter from response headers if available
                    if self.rate_limiter and hasattr(response, 'headers'):
                        self.rate_limiter.update_from_headers(
                            dict(response.headers) if response.headers else {},
                            self.header_mapping
                        )
                        self.rate_limiter.reset_429_counter()

                    # Extract embeddings - response.embeddings is a list of embedding objects
                    embeddings = [emb.values for emb in response.embeddings]

                    # If the response is empty, raise an error
                    if not embeddings:
                        raise ValueError("Received empty embeddings from Google Gemini.")

                    # If multiple chunks, average the embeddings
                    if len(embeddings) > 1:
                        print(f"Info: Received {len(embeddings)} embeddings from Google. Averaging them.")
                        # Calculate weighted average by text length
                        weights = [len(chunk) for chunk in input_chunks]
                        avg_embedding = np.average(embeddings, axis=0, weights=weights)
                        # Normalize the embedding
                        avg_embedding = avg_embedding / np.linalg.norm(avg_embedding)
                        return avg_embedding.tolist()
                    else:
                        # Single embedding - ensure it's normalized for consistency
                        embedding = np.array(embeddings[0])
                        embedding = embedding / np.linalg.norm(embedding)
                        return embedding.tolist()

                except Exception as e:
                    # Check if it's a 429 error
                    if hasattr(e, 'status_code') and e.status_code == 429:
                        headers = dict(e.response.headers) if hasattr(e, 'response') and hasattr(e.response, 'headers') else {}
                        if self.rate_limiter:
                            await self.rate_limiter.handle_429(headers)
                        # Don't count this as a retry attempt
                        continue

                    if attempt < retry_count - 1:
                        await asyncio.sleep(2 ** attempt)  # Exponential backoff
                    else:
                        raise

    def get_identifier(self) -> str:
        """Return a unique identifier for this provider and model."""
        task_type = self.config.get('task_type', 'default')
        return f"{self.config['provider']}_{self.config['model']}_{task_type}"

    def get_tokenizer(self, text: str) -> List[int]:
        """Tokenize the input text using Google's tokenizer."""
        # Use the count_tokens API for token estimation
        if not self.client:
            asyncio.run(self.initialize())

        try:
            tokenizer = genai.LocalTokenizer(model_name=self.config["model"])
            result = tokenizer.compute_tokens(text)
            # Return empty list as we don't have actual token IDs, just counts
            # This is just for compatibility with the interface
            return result
        except Exception:
            return []

    def chunk_text(self, text: str) -> List[str]:
        """Chunk text if it exceeds the context window."""
        # Use character-based approximation for chunking
        # Approximate 4 characters per token for English text
        avg_chars_per_token = 4
        max_chars = self.config["context_limit"] * avg_chars_per_token
        
        # If text is short enough, return as single item
        if len(text) <= max_chars:
            return [text]
        
        # Need to chunk - split by characters
        chunks = []
        for i in range(0, len(text), max_chars):
            chunk = text[i:i + max_chars]
            chunks.append(chunk)
        
        return chunks

def create_provider(config: Dict[str, Any], shared_rate_limiter=None) -> EmbeddingProvider:
    """Factory function to create the appropriate provider."""
    provider_name = config["api_spec"]
    api_key = api_keys[config["provider"]]

    if provider_name == "openai":
        return OpenAIProvider(config, api_key, shared_rate_limiter)
    elif provider_name == "cohere":
        return CohereProvider(config, api_key, shared_rate_limiter)
    elif provider_name == "google":
        return GoogleProvider(config, api_key, shared_rate_limiter)
    else:
        raise ValueError(f"Unknown provider: {provider_name}")

## Embedding Generation Functions

In [10]:
nest_asyncio.apply()

async def retrieve_embeddings(
    provider: EmbeddingProvider,
    semaphore: TrackedSemaphore,
    doc_id: str,
    content: str,
    stats: EmbeddingStatistics
) -> Dict[str, Dict[str, List[float]]]:
    """Make an async embedding call for a specific document."""
    provider_name = provider.get_identifier()

    try:
        embeddings = await provider.create_embedding(content, semaphore)
        
        # Check if embeddings are empty or invalid
        if not embeddings or len(embeddings) == 0:
            stats.record_failure(doc_id, "Empty embeddings returned")
            logger.error(f"Empty embeddings for document {doc_id} with provider {provider_name}")
            return {doc_id: {provider_name: []}}

        stats.record_success(doc_id)
        return {doc_id: {provider_name: embeddings}}

    except Exception as e:
        error_msg = str(e)
        
        # Check if it's a rate limit error
        if hasattr(e, 'status_code') and e.status_code == 429:
            stats.record_rate_limit()
            logger.warning(f"Rate limit (429) for document {doc_id} with provider {provider_name}")
        
        stats.record_failure(doc_id, error_msg)
        logger.error(f"Error creating embedding for document {doc_id} with provider {provider_name}: {error_msg}")
        return {doc_id: {provider_name: []}}

async def control_embeddings_retrieval(
    provider: EmbeddingProvider,
    semaphore: TrackedSemaphore,
    dfi: pl.DataFrame,
    progress: rich.progress.Progress,
    task_id: rich.progress.TaskID,
    stats: EmbeddingStatistics
) -> Dict[str, Dict[str, List[float]]]:
    """Control async embedding calls for a specific provider."""

    results = {}

    async def process_row(row):
        res = await retrieve_embeddings(provider, semaphore, row["url"], row["content"], stats)
        progress.advance(task_id)
        return res

    tasks = [asyncio.create_task(process_row(row)) for row in dfi.rows(named=True)]
    completed = await asyncio.gather(*tasks, return_exceptions=True)

    # Combine all results into one dictionary
    for res in completed:
        if not(isinstance(res, BaseException)):
            for doc_id, providers_dict in res.items():
                if doc_id not in results:
                    results[doc_id] = {}
                results[doc_id].update(providers_dict)

    return results

async def get_embeddings_for_provider(
    provider: EmbeddingProvider,
    semaphore: TrackedSemaphore,
    dfi: pl.DataFrame,
    progress: rich.progress.Progress,
    task_id: rich.progress.TaskID,
    stats: EmbeddingStatistics
) -> Dict[str, Dict[str, List[float]]]:
    """Get embeddings for a specific provider, using cache when available."""

    provider_name = provider.get_identifier()
    stats.start()

    # No embeddings file exists: create embeddings
    if not os.path.exists(file_path_out_embeddings_pickle):
        embeddings = await control_embeddings_retrieval(
            provider, semaphore, dfi, progress, task_id, stats
        )
        stats.complete()
        return embeddings

    # Embeddings file exists: check if we have embeddings for this provider
    else:
        logger.info(f"Checking cached embeddings for {provider_name}...")
        with open(file_path_out_embeddings_pickle, "rb") as fIn:
            cache_data = pickle.load(fIn)

        # Check if we have this provider's embeddings
        if provider_name in cache_data:
            logger.info(f"Found cached embeddings for {provider_name}")
            cached_docs = set(cache_data[provider_name].keys())
            needed_docs = set(dfi["url"])

            # All documents already have embeddings
            if cached_docs.issuperset(needed_docs):
                logger.info(f"Using all cached embeddings for {provider_name} ({len(needed_docs)} documents)")
                
                # Mark all as success since they're cached
                for doc_id in needed_docs:
                    stats.record_success(doc_id)
                
                stats.complete()
                return {doc_id: {provider_name: cache_data[provider_name][doc_id]} 
                        for doc_id in needed_docs}

            # Some documents need to be processed
            else:
                missing_docs = needed_docs - cached_docs
                logger.info(f"Processing {len(missing_docs)} missing documents for {provider_name} (cached: {len(needed_docs - missing_docs)})")

                # Mark cached docs as success
                for doc_id in (needed_docs - missing_docs):
                    stats.record_success(doc_id)

                # Process only missing documents
                missing_df = dfi.filter(pl.col("url").is_in(list(missing_docs)))
                new_embeddings = await control_embeddings_retrieval(
                    provider, semaphore, missing_df, progress, task_id, stats
                )

                # Combine cached with new embeddings
                combined = {}
                for doc_id in needed_docs:
                    combined[doc_id] = {}
                    if doc_id in cache_data[provider_name]:
                        combined[doc_id][provider_name] = cache_data[provider_name][doc_id]
                    elif doc_id in new_embeddings:
                        combined[doc_id].update(new_embeddings[doc_id])

                # Update cache
                cache_data[provider_name].update({
                    doc_id: new_embeddings[doc_id][provider_name] 
                    for doc_id in new_embeddings 
                    if provider_name in new_embeddings[doc_id]
                })
                save_dict_pickle(cache_data, file_path_out_embeddings_pickle)

                stats.complete()
                return combined

        # Provider not in cache
        else:
            logger.info(f"No cached embeddings found for {provider_name}, processing all {len(dfi)} documents")
            new_embeddings = await control_embeddings_retrieval(
                provider, semaphore, dfi, progress, task_id, stats
            )

            # Add new provider to cache
            cache_data[provider_name] = {
                doc_id: new_embeddings[doc_id][provider_name]
                for doc_id in new_embeddings
                if provider_name in new_embeddings[doc_id]
            }
            save_dict_pickle(cache_data, file_path_out_embeddings_pickle)

            stats.complete()
            return new_embeddings

async def get_all_embeddings(
    providers: List[EmbeddingProvider],
    dfi: pl.DataFrame
) -> tuple[Dict[str, Dict[str, List[float]]], Dict[str, EmbeddingStatistics]]:
    """Get embeddings from all specified providers and return statistics."""

    # Create statistics trackers for each provider
    stats_dict = {p.get_identifier(): EmbeddingStatistics(p.get_identifier()) for p in providers}

    # Create a single shared Progress instance
    progress = rich.progress.Progress(
        rich.progress.TextColumn("[bold blue]{task.description}"),
        rich.progress.BarColumn(bar_width=None),
        rich.progress.TimeElapsedColumn(),
        refresh_per_second=2,
        transient=False
    )

    # In get_all_embeddings():
    task_ids = {p.get_identifier(): progress.add_task(p.get_identifier(), total=len(dfi)) for p in providers}

    # Initialize semaphores for each provider
    semaphores = {p.get_identifier(): TrackedSemaphore(p.config["concurrent_requests"]) for p in providers}

    try:
        with progress:
            # Dispatch tasks for all providers using the shared progress instance
            tasks = [
                get_embeddings_for_provider(
                    p,
                    semaphores[p.get_identifier()],
                    dfi,
                    progress,
                    task_ids[p.get_identifier()],
                    stats_dict[p.get_identifier()]
                ) for p in providers
            ]
            # collect results
            provider_results = await asyncio.gather(*tasks, return_exceptions=True)
    finally:
        progress.stop()


    # Combine results from all providers
    all_embeddings = {}
    for r in provider_results:
        if isinstance(r, BaseException):
            logger.error(f"Error occurred while processing provider results: {r}")
            print(f"Error occurred while processing provider results: {r}")
        else:
            for doc_id, provider_dict in r.items():
                all_embeddings.setdefault(doc_id, {}).update(provider_dict)

    logger.info("All embedding tasks completed")
    print("All tasks completed.")
    return all_embeddings, stats_dict

## Load Data

In [11]:
# Load and prepare document data
print(f"Loading documents from {file_path_in_text} ...")
docs = pl.read_csv(file_path_in_text)
print(f"Loaded {len(docs)} documents.")

# Check for required columns
required_columns = ["url", "content"]
for col in required_columns:
    if col not in docs.columns:
        raise ValueError(f"Missing required column: {col}")

# Ensure content is a string
docs = docs.with_columns(pl.col("content").cast(pl.Utf8))

# Filter too short documents
if min_tokens > 0:
    encoding = tiktoken.get_encoding("cl100k_base")
    # Filter by token count
    docs.filter(
        pl.col("content")
        .map_elements(lambda x: len(encoding.encode(x)), return_dtype=pl.Int32)
        >= min_tokens
    )
    print(f"Filtered to {len(docs)} documents with at least {min_tokens} tokens")

# Limit number of documents if specified
if max_documents > 0:
    docs = docs.head(max_documents)
    print(f"Limited to {len(docs)} documents")

# Display basic stats
print(f"Processing {len(docs)} documents")

Loading documents from ./in-data/corpus_20260111.csv ...
Loaded 196894 documents.
Filtered to 196894 documents with at least 10 tokens
Processing 196894 documents


## Main Execution Block

### Create Shared Rate Limiters

Create shared rate limiters per platform provider. This ensures that multiple models hosted by the same provider (e.g., different models on Academic Cloud) share the same rate limiting constraints.

In [12]:
# Create shared rate limiters per platform provider
platform_rate_limiters = {}

for provider_name in active_providers:
    config = providers_config[provider_name]
    platform = config["provider"]
    
    # Create rate limiter once per platform
    if platform not in platform_rate_limiters:
        if "rate_limits" in config and config["rate_limits"]:
            windows = {}
            for window_name, limit in config["rate_limits"].items():
                # Map window names to seconds
                window_seconds = {
                    "minute": 60,
                    "hour": 3600,
                    "day": 86400,
                    "month": 2592000  # 30 days
                }
                if window_name in window_seconds:
                    windows[window_name] = (limit, window_seconds[window_name])

            time_restrictions = config.get("time_restrictions", None)
            safety_margin = config.get("safety_margin", 0.95)
            
            platform_rate_limiters[platform] = HeaderBasedRateLimiter(
                windows=windows,
                time_restrictions=time_restrictions,
                safety_margin=safety_margin
            )
        else:
            # Fallback to simple per-minute rate limiting
            rate_limit = config.get("rate_limit_per_minute", 1000)
            platform_rate_limiters[platform] = HeaderBasedRateLimiter(
                windows={"minute": (rate_limit, 60)},
                time_restrictions=None,
                safety_margin=0.95
            )

print(f"Created {len(platform_rate_limiters)} shared rate limiters for platforms:")
for platform, limiter in platform_rate_limiters.items():
    print(f"  - {platform}: {list(limiter.buckets.keys())}")

Created 4 shared rate limiters for platforms:
  - openai: ['minute', 'hour']
  - academic-cloud: ['minute', 'hour', 'day', 'month']
  - cohere: ['minute']
  - google: ['minute']


**How Rate Limiting Works Now:**

1. **Platform-Level Sharing**: Models from the same platform provider (e.g., `chat-ai-e5-mistral`, `chat-ai-multilingual-e5`, and `chat-ai-qwen3` all using `academic-cloud`) share a single rate limiter.

2. **Model-Level Stats**: Statistics and failure tracking remain separate for each model, so you can see which specific model succeeded or failed.

3. **Example**: If you have three Academic Cloud models configured with a 60 req/min limit, all three models will collectively be limited to 60 requests per minute (not 60 each), but you'll still see separate success/failure counts for each model.

In [13]:
# Initialize global tracking variables
uploaded_ids = {}

# Initialize providers with shared rate limiters
providers = [
    create_provider(
        providers_config[provider_name],
        shared_rate_limiter=platform_rate_limiters[providers_config[provider_name]["provider"]]
    )
    for provider_name in active_providers
]

print(f"\nInitialized {len(providers)} providers:")
for provider in providers:
    provider_name = provider.get_identifier()
    platform = provider.config["provider"]
    print(f"  - {provider_name} (platform: {platform})")

# Get embeddings from all configured providers
embeddings_dict, stats_dict = asyncio.run(get_all_embeddings(providers, docs))

Output()

2026-01-18 20:12:20,220 - __main__ - INFO - Provider openai_text-embedding-3-small started processing



Initialized 6 providers:
  - openai_text-embedding-3-small (platform: openai)
  - academic-cloud_e5-mistral-7b-instruct (platform: academic-cloud)
  - academic-cloud_multilingual-e5-large-instruct (platform: academic-cloud)
  - academic-cloud_qwen3-embedding-4b (platform: academic-cloud)
  - cohere_embed-v4.0_clustering (platform: cohere)
  - google_gemini-embedding-001_SEMANTIC_SIMILARITY (platform: google)


2026-01-18 20:12:25,731 - __main__ - INFO - Provider academic-cloud_e5-mistral-7b-instruct started processing


2026-01-18 20:12:29,618 - __main__ - INFO - Provider academic-cloud_multilingual-e5-large-instruct started processing


...

Outside allowed time window. Waiting 5 minutes...

2026-01-20 19:52:02,606 - __main__ - INFO - Provider google_gemini-embedding-001_SEMANTIC_SIMILARITY completed processing


Outside allowed time window. Waiting 5 minutes...

2026-01-20 21:42:48,922 - __main__ - WARNING - Provider academic-cloud_e5-mistral-7b-instruct - Document None failed: You need to specify either `text` or `text_target`.
2026-01-20 21:42:48,928 - __main__ - ERROR - Error creating embedding for document None with provider academic-cloud_e5-mistral-7b-instruct: You need to specify either `text` or `text_target`.


KeyboardInterrupt: 

## Embedding Generation Statistics

Summary of embedding generation results per provider.

In [15]:
# Display per-provider statistics
print("\n" + "="*80)
print("Provider Statistics:")
print("="*80)

for provider in providers:
    provider_name = provider.get_identifier()
    if provider_name in stats_dict:
        stats_dict[provider_name].print_summary()

print("\n" + "="*80)


Provider Statistics:


NameError: name 'stats_dict' is not defined

### Failure Analysis

Analyze which documents failed for which providers.

In [ ]:
# Analyze failures across providers
all_doc_ids = set(docs["url"])
provider_names = [p.get_identifier() for p in providers]

# Track which docs failed for each provider
failed_by_provider = {}
for provider_name in provider_names:
    if provider_name in stats_dict:
        failed_docs = [f["doc_id"] for f in stats_dict[provider_name].failed_docs]
        failed_by_provider[provider_name] = set(failed_docs)

# Find docs that failed for ALL providers
if failed_by_provider:
    critical_failures = set.intersection(*failed_by_provider.values()) if len(failed_by_provider) > 0 else set()
else:
    critical_failures = set()

# Find docs that succeeded for at least one provider
docs_with_success = all_doc_ids - critical_failures if critical_failures else all_doc_ids

# Find docs with partial success (failed for some providers)
docs_with_partial_success = set()
for doc_id in all_doc_ids:
    failed_count = sum(1 for failed_set in failed_by_provider.values() if doc_id in failed_set)
    if 0 < failed_count < len(provider_names):
        docs_with_partial_success.add(doc_id)

print("\n" + "="*80)
print("Overall Summary:")
print("="*80)
print(f"Total documents: {len(all_doc_ids)}")
print(f"Critical failures (failed for all providers): {len(critical_failures)}")
print(f"Documents with partial success: {len(docs_with_partial_success)}")
print(f"Documents with full success: {len(docs_with_success) - len(docs_with_partial_success)}")
print("="*80)

# Show critical failures if any
if critical_failures:
    print(f"\nDocuments that failed for ALL providers:")
    for doc_id in list(critical_failures)[:10]:  # Show first 10
        print(f"  - {doc_id}")
    if len(critical_failures) > 10:
        print(f"  ... and {len(critical_failures) - 10} more")

# Show per-provider failures
if docs_with_partial_success:
    print(f"\nPer-provider failures:")
    for provider_name, failed_set in failed_by_provider.items():
        if failed_set:
            print(f"  {provider_name}: {len(failed_set)} failures")

### Export Statistics

Save statistics to JSON file for later analysis.

In [ ]:
# Export statistics to JSON
# Build statistics export structure
stats_export = {
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "total_documents": len(all_doc_ids),
    "providers": {},
    "summary": {
        "critical_failures": len(critical_failures),
        "partial_success": len(docs_with_partial_success),
        "full_success": len(docs_with_success) - len(docs_with_partial_success),
        "critical_failure_docs": list(critical_failures)
    }
}

# Add per-provider statistics
for provider in providers:
    provider_name = provider.get_identifier()
    if provider_name in stats_dict:
        stats_export["providers"][provider_name] = stats_dict[provider_name].to_dict()

# Save to JSON
print(f"\nSaving statistics to {file_path_out_stats_json}...")
with open(file_path_out_stats_json, 'w') as f:
    json.dump(stats_export, f, indent=2)
print(f"Statistics saved successfully.")

### Rate Limiter Status Monitoring

This section provides tools to monitor the rate limiter status across all providers.

In [ ]:
# Display rate limiter status for all providers
def display_rate_limiter_status(providers):
    """Display current rate limiter status for all providers."""
    console = rich.console.Console()

    console.print("\n[bold cyan]Rate Limiter Status[/bold cyan]\n")

    for provider in providers:
        provider_name = provider.get_identifier()
        console.print(f"[bold]{provider_name}[/bold]")
        
        if provider.rate_limiter:
            status = provider.rate_limiter.get_status()
            
            # Show time restrictions
            if provider.rate_limiter.time_restrictions:
                start, end = provider.rate_limiter.time_restrictions
                is_allowed = provider.rate_limiter._is_allowed_time()
                status_text = "[green]✓ Allowed[/green]" if is_allowed else "[red]✗ Restricted[/red]"
                console.print(f"  Time window: {start:02d}:00-{end:02d}:00 UTC {status_text}")
            
            # Show bucket status
            for window_name, tokens in status.items():
                bucket = provider.rate_limiter.buckets[window_name]
                capacity = bucket.capacity
                percentage = (tokens / capacity) * 100
                
                # Color based on percentage
                if percentage > 50:
                    color = "green"
                elif percentage > 20:
                    color = "yellow"
                else:
                    color = "red"
                
                console.print(f"  {window_name}: [{color}]{tokens:.1f}/{capacity:.0f}[/{color}] ({percentage:.1f}%)")
            
            # Show 429 status
            if provider.rate_limiter.consecutive_429s > 0:
                console.print(f"  [red]Recent 429s: {provider.rate_limiter.consecutive_429s}[/red]")
        else:
            console.print("  [dim]No rate limiter configured[/dim]")
        
        console.print()

# Uncomment to display status after providers are initialized
# display_rate_limiter_status(providers)


#### Usage Example:

To monitor rate limiter status during processing:

```python
# Display current status
display_rate_limiter_status(providers)
```

The output will show:
- Provider name and model
- Time window restrictions and current status
- Token counts for each window (minute/hour/day/month)
- Percentage of available quota remaining
- Color-coded status (green >50%, yellow >20%, red <20%)
- Recent 429 errors if any

You can run this at any time to check the current state of the rate limiters.


## Diagnostics, Reporting and Saving

In [ ]:
embeddings_dict

In [ ]:
# Merge embeddings to dataframe - create columns for each provider
for provider in providers:
    provider_name = provider.get_identifier()
    column_name = f"embeddings_{provider_name}"

    print(f"Adding {column_name} to dataframe...")
    # Create a mapping function that extracts the right embedding
    def get_embedding(doc_id, p_id=provider_name):
        if doc_id in embeddings_dict and p_id in embeddings_dict[doc_id]:
            return embeddings_dict[doc_id][p_id]
        return None

    docs = docs.with_columns(
        pl.col("url")
        .map_elements(lambda x: get_embedding(x, provider_name), return_dtype=pl.List(pl.Float32))
        .alias(column_name)
    )

In [ ]:
# Show first few documents and their providers
print("Sample of embeddings:")
sample_doc_id = list(embeddings_dict.keys())[0]
print(f"Document ID: {sample_doc_id}")
print(f"Available embedding providers: {list(embeddings_dict[sample_doc_id].keys())}")

# Display the enriched DataFrame
print("\nFirst rows of enriched docs dataframe:")
docs.head(3)

## Saving Data

In [ ]:
# Ensure output directory exists
os.makedirs(file_path_out, exist_ok=True)

# === Save just the embeddings to disk separately ===

# Prepare the cache data format for just the embeddings
cache_data = {}
for provider in providers:
    provider_name = provider.get_identifier()
    cache_data[provider_name] = {
        doc_id: embeddings_dict[doc_id][provider_name]
        for doc_id in embeddings_dict
        if provider_name in embeddings_dict[doc_id]
    }

# - pickle format
print(f"Saving {file_path_out_embeddings_pickle} to disk ...")
save_dict_pickle(cache_data, file_path_out_embeddings_pickle)

# - parquet format
print(f"Saving {file_path_out_embeddings_parquet} to disk ...")
save_dict_parquet(cache_data, file_path_out_embeddings_parquet)


# === Save the configuration to disk ===

processing_metadata = {
    "processing_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "documents": {
        "total_available": len(docs),
        "processed": len(docs) if max_documents == -1 else max_documents,
        "min_tokens_threshold": min_tokens
    },
    "providers": {
        p.get_identifier(): {
            "provider": p.config["provider"],
            "model": p.config["model"],
            "api_spec": p.config.get("api_spec", None),
            "base_url": p.config.get("base_url", None),
            "dimensions": p.config.get("dimensions", None),
            "embedding_type": p.config.get("embedding_type", None),
            "embedding_types": p.config.get("embedding_types", None),
            "input_type": p.config.get("input_type", None),
            "encoding": p.config.get("encoding", None),
            "context_limit": p.config.get("context_limit", None),
            "max_texts_per_call": p.config.get("max_texts_per_call", None),
            "rate_limit_per_minute": p.config.get("rate_limit_per_minute", None),
            "concurrent_requests": p.config.get("concurrent_requests", None),
            "documents_processed": len(cache_data.get(p.get_identifier(), {}))
        }
        for p in providers
    }
}
save_metadata_json(processing_metadata, file_path_out_config_json)


# === Save the full docs dataframe to disk ===

# - pickle format
print(f"Saving {file_path_out_docs_pickle} to disk ...")
save_df_pickle(docs, file_path_out_docs_pickle)

# - parquet format
print(f"Saving {file_path_out_docs_parquet} to disk ...")
save_df_parquet(docs, file_path_out_docs_parquet)

# - csv format (but without the embeddings, as they are too large for csv)
print(f"Saving {file_path_out_docs_csv} to disk ...")
docs_for_csv = docs.clone().drop(pl.selectors.starts_with("embeddings_"))
docs_for_csv.write_csv(file_path_out_docs_csv)
print("Saving done.")

In [ ]:
print("\nAll done.\n")